In [36]:
from skimage.metrics import structural_similarity as ssim_func

In [40]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture
from skimage.metrics import structural_similarity as ssim
from skimage.measure import shannon_entropy
import numpy as np
from sklearn.cluster import KMeans

## QImT_GM: Quantum Image Transformation with Gaussian Mixtures.
class QImT_GM:

    def __init__(self, image_path):

        img = cv2.imread(image_path)

        if img is None:
            raise ValueError("Image not found")

        self.original = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        self.reconstructed = None

        self.mus = None
        self.sigmas = None
        self.weights = None

        self.kraus_ops = None


    def run_segmentation(self, k_clusters, sharpening_gamma=2.0):

        pixels = self.original.reshape(-1, 1)

        gmm = GaussianMixture(n_components=k_clusters, random_state=0)
        gmm.fit(pixels)

        gmm = GaussianMixture(
            n_components=k_clusters,
            covariance_type='full',
            random_state=0
        )

        gmm.fit(pixels)

        self.mus = gmm.means_.flatten()
        self.sigmas = np.sqrt(gmm.covariances_.flatten())
        self.weights = gmm.weights_
        order = np.argsort(self.mus)

        self.mus = self.mus[order]
        self.sigmas = self.sigmas[order]
        self.weights = self.weights[order]

        intensity = np.arange(256)

        G = []

        for mu, sigma, w in zip(self.mus, self.sigmas, self.weights):

            gaussian = w * np.exp(
                -(intensity - mu)**2 / (2 * sigma**2)
            )

            G.append(gaussian)

        G = np.array(G)


        G_norm = G / np.sum(G, axis=0)


        G_norm = G_norm ** sharpening_gamma
        G_norm = G_norm / np.sum(G_norm, axis=0)


        kraus_ops = []

        for g in G_norm:

            M = np.diag(np.sqrt(g))

            kraus_ops.append(M)

        self.kraus_ops = kraus_ops

        prob_maps = []

        for M in kraus_ops:

            diag_elements = np.diag(M @ M.T)

            prob = diag_elements[self.original]

            prob_maps.append(prob)

        prob_maps = np.array(prob_maps)


        reconstructed = np.zeros_like(self.original, dtype=np.float64)

        for k, mu in enumerate(self.mus):

            reconstructed += mu * prob_maps[k]

        self.reconstructed = np.clip(reconstructed,0,255).astype(np.uint8)


        print("EM segmentation finished")
        print("Means:", self.mus)


    def calculate_metrics(self):

        mse = np.mean(
        (self.original.astype(np.float64)
        - self.reconstructed.astype(np.float64))**2
        )

        psnr = 10 * np.log10((255**2)/mse) if mse != 0 else float('inf')

        ssim_val = ssim_func(self.original, self.reconstructed, data_range=255)

        ent_orig = shannon_entropy(self.original)
        ent_rec = shannon_entropy(self.reconstructed)

        ent_change = ent_rec - ent_orig
        percent_change = (ent_change/ent_orig)*100 if ent_orig != 0 else 0

        return {
          "PSNR": round(psnr,2),
          "SSIM": round(ssim_val,4),
          "Entropy Change": round(ent_change,4),
          "Percent Change": round(percent_change,2)
        }


    def plot_images(self):

        plt.figure()
        plt.subplot(1,2,1)
        plt.title("Original")
        plt.imshow(self.original, cmap='gray')
        plt.axis('off')
        plt.subplot(1,2,2)

        plt.title(f"K={len(self.mus)}")
        plt.imshow(self.reconstructed, cmap='gray')
        plt.axis('off')

        plt.tight_layout()
        plt.show()


    def plot_results(self):

        metrics = self.calculate_metrics()

        plt.figure(figsize=(12,5))

        plt.hist(self.original.ravel(),
                 bins=256,
                 range=(0,256),
                 alpha=0.5,
                 label='Original')

        plt.hist(self.reconstructed.ravel(),
                 bins=256,
                 range=(0,256),
                 alpha=0.5,
                 label='Reconstructed')

        plt.title(
            f"K={len(self.mus)} | "
            f"PSNR={metrics['PSNR']} | "
            f"SSIM={metrics['SSIM']} | "
            f"%Entropy Change ={metrics['Percent Change']:.2f}%"
        )

        plt.legend()

        plt.tight_layout()
        plt.show()


In [34]:
## QImT: Quantum Image Transformation.
class QImT:

    def __init__(self, image_path):

        img = cv2.imread(image_path)

        if img is None:
            raise ValueError("Image not found.")

        self.original = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        self.reconstructed = None
        self.mus = None
        self.kraus_ops = None


    def run_segmentation(self, k_clusters, sharpening_gamma=2.0):

        pixels = self.original.reshape(-1,1)

        kmeans = KMeans(
            n_clusters=k_clusters,
            random_state=0,
            n_init=10
        )

        kmeans.fit(pixels)

        self.mus = np.sort(kmeans.cluster_centers_.flatten())


        # Delta parameter
        global_std = np.std(self.original)
        delta = max(10, 0.5 * global_std)


        # Build Gaussian POVM first
        intensity = np.arange(256)

        G = np.array([
            np.exp(-(intensity - mu)**2 / (2 * delta**2))
            for mu in self.mus
        ])

        G_norm = G / np.sum(G, axis=0)
        G_norm = G_norm ** sharpening_gamma
        G_norm = G_norm / np.sum(G_norm, axis=0)


        # Construct Kraus operators
        kraus_ops = []

        for g in G_norm:
            M = np.diag(np.sqrt(g))
            kraus_ops.append(M)

        self.kraus_ops = kraus_ops


        # Compute probabilities from Kraus operators
        prob_maps = []

        for M in kraus_ops:
            diag_elements = np.diag(M.conj().T @ M)
            prob = diag_elements[self.original]
            prob_maps.append(prob)

        prob_maps = np.array(prob_maps)


        reconstructed = np.zeros_like(self.original, dtype=np.float64)

        for k, mu in enumerate(self.mus):

            reconstructed += mu * prob_maps[k]

        self.reconstructed = np.clip(reconstructed,0,255).astype(np.uint8)

        #print("Cluster means:", self.mus)


    def calculate_metrics(self):

        mse = np.mean(
        (self.original.astype(np.float64)
        - self.reconstructed.astype(np.float64))**2
        )

        psnr = 10 * np.log10((255**2)/mse) if mse != 0 else float('inf')

        ssim_val = ssim_func(self.original, self.reconstructed, data_range=255)

        ent_orig = shannon_entropy(self.original)
        ent_rec = shannon_entropy(self.reconstructed)

        ent_change = ent_rec - ent_orig
        percent_change = (ent_change/ent_orig)*100 if ent_orig != 0 else 0

        return {
          "PSNR": round(psnr,2),
          "SSIM": round(ssim_val,4),
          "Entropy Change": round(ent_change,4),
          "Percent Change": round(percent_change,2)
        }

    def plot_images(self):

        plt.figure()
        plt.subplot(1,2,1)
        plt.title("Original")
        plt.imshow(self.original, cmap='gray')
        plt.axis('off')
        plt.subplot(1,2,2)

        plt.title(f"K={len(self.mus)}")
        plt.imshow(self.reconstructed, cmap='gray')
        plt.axis('off')

        plt.tight_layout()
        plt.show()


    def plot_results(self):

        metrics = self.calculate_metrics()

        plt.hist(self.original.ravel(),
                 bins=256,
                 range=(0,256),
                 alpha=0.5,
                 label='Original')

        plt.hist(self.reconstructed.ravel(),
                 bins=256,
                 range=(0,256),
                 alpha=0.5,
                 label='Reconstructed')

        plt.title(
            f"K={len(self.mus)} | "
            f"PSNR={metrics['PSNR']} | "
            f"SSIM={metrics['SSIM']} | "
            f"%Entropy Change={metrics['Percent Change']:.4f}%"
        )

        plt.legend()

        plt.tight_layout()
        plt.show()

In [11]:
import kagglehub

# Download dataset
path = kagglehub.dataset_download("balraj98/berkeley-segmentation-dataset-500-bsds500")

print("Dataset path:", path)

Using Colab cache for faster access to the 'berkeley-segmentation-dataset-500-bsds500' dataset.
Dataset path: /kaggle/input/berkeley-segmentation-dataset-500-bsds500


In [12]:
import os

for root, dirs, files in os.walk(path):
    print("Current path:", root)
    print("Folders:", dirs)
    print("Sample files:", files[:3])
    print("-" * 40)

Current path: /kaggle/input/berkeley-segmentation-dataset-500-bsds500
Folders: ['ground_truth', 'images']
Sample files: []
----------------------------------------
Current path: /kaggle/input/berkeley-segmentation-dataset-500-bsds500/ground_truth
Folders: ['val', 'test', 'train']
Sample files: []
----------------------------------------
Current path: /kaggle/input/berkeley-segmentation-dataset-500-bsds500/ground_truth/val
Folders: []
Sample files: ['210088.mat', '175032.mat', '220075.mat']
----------------------------------------
Current path: /kaggle/input/berkeley-segmentation-dataset-500-bsds500/ground_truth/test
Folders: []
Sample files: ['176051.mat', '159022.mat', '160006.mat']
----------------------------------------
Current path: /kaggle/input/berkeley-segmentation-dataset-500-bsds500/ground_truth/train
Folders: []
Sample files: ['15088.mat', '135069.mat', '376001.mat']
----------------------------------------
Current path: /kaggle/input/berkeley-segmentation-dataset-500-bsds50

In [15]:
import os

train_path = None

for root, dirs, files in os.walk(path):
    jpg_files = [f for f in files if f.endswith(".jpg")]

    # look for a folder that actually contains images
    if jpg_files and "train" in root.lower():
        train_path = root
        print("Found train folder:", train_path)
        print("Sample files:", jpg_files[:5])
        break

if train_path is None:
    print("Train folder not found")

✅ Found train folder: /kaggle/input/berkeley-segmentation-dataset-500-bsds500/images/train
Sample files: ['374067.jpg', '100098.jpg', '60079.jpg', '198054.jpg', '45077.jpg']


In [17]:
import os

print(os.listdir("/kaggle/input/berkeley-segmentation-dataset-500-bsds500"))

['ground_truth', 'images']


In [28]:
import os
import random

base_path = "/kaggle/input/berkeley-segmentation-dataset-500-bsds500/images/train"

all_images = [os.path.join(base_path, f) for f in os.listdir(base_path) if f.endswith(".jpg")]

# pick 50 random images
sample_images = random.sample(all_images, 100)

print("Sample size:", len(sample_images))

Sample size: 100


In [37]:
# metric lists
psnr_list = []
ssim_list = []
entropy_change_list = []

for idx, image_path in enumerate(sample_images):
    #try:
        print(f"Processing {idx+1}/100")

        qimt = QImT(image_path)
        qimt.run_segmentation(k_clusters=8, sharpening_gamma=2.0)

        metrics = qimt.calculate_metrics()

        # append to lists
        psnr_list.append(metrics.get("PSNR"))
        ssim_list.append(metrics.get("SSIM"))
        entropy_change_list.append(metrics.get("Percent Change"))

Processing 1/100
Processing 2/100
Processing 3/100
Processing 4/100
Processing 5/100
Processing 6/100
Processing 7/100
Processing 8/100
Processing 9/100
Processing 10/100
Processing 11/100
Processing 12/100
Processing 13/100
Processing 14/100
Processing 15/100
Processing 16/100
Processing 17/100
Processing 18/100
Processing 19/100
Processing 20/100
Processing 21/100
Processing 22/100
Processing 23/100
Processing 24/100
Processing 25/100
Processing 26/100
Processing 27/100
Processing 28/100
Processing 29/100
Processing 30/100
Processing 31/100
Processing 32/100
Processing 33/100
Processing 34/100
Processing 35/100
Processing 36/100
Processing 37/100
Processing 38/100
Processing 39/100
Processing 40/100
Processing 41/100
Processing 42/100
Processing 43/100
Processing 44/100
Processing 45/100
Processing 46/100
Processing 47/100
Processing 48/100
Processing 49/100
Processing 50/100
Processing 51/100
Processing 52/100
Processing 53/100
Processing 54/100
Processing 55/100
Processing 56/100
P

In [38]:
def summarize(arr):
    return np.mean(arr), np.std(arr)

psnr_mean, psnr_std = summarize(psnr_list)
ssim_mean, ssim_std = summarize(ssim_list)
ent_mean, ent_std = summarize(entropy_change_list)

print("\n===== POVM Kmeans =====")
print(f"PSNR: {psnr_mean:.4f} ± {psnr_std:.4f}")
print(f"SSIM: {ssim_mean:.4f} ± {ssim_std:.4f}")
print(f"Entropy Change (%): {ent_mean:.4f} ± {ent_std:.4f}")


===== POVM Kmeans =====
PSNR: 34.7968 ± 2.8695
SSIM: 0.9783 ± 0.0164
Entropy Change (%): -6.8283 ± 3.1276


In [41]:
# metric lists
psnr_list_ = []
ssim_list_ = []
entropy_change_list_ = []

for idx, image_path in enumerate(sample_images):
    try:
        print(f"Processing {idx+1}/100")

        qimt = QImT_GM(image_path)
        qimt.run_segmentation(k_clusters=8, sharpening_gamma=2.0)

        metrics = qimt.calculate_metrics()

        # append to lists
        psnr_list_.append(metrics.get("PSNR"))
        ssim_list_.append(metrics.get("SSIM"))
        entropy_change_list_.append(metrics.get("Percent Change"))

    except Exception as e:
        print(f"Error with {image_path}: {e}")

Processing 1/100
EM segmentation finished
Means: [ 34.95432071  54.88680476  81.10589777 104.43862484 129.93456337
 161.86921599 196.66409424 232.17393364]
Processing 2/100
EM segmentation finished
Means: [ 14.75837494  32.38761659  50.23430882  72.61277927  96.76896509
 128.32970187 178.51701228 232.38386972]
Processing 3/100
EM segmentation finished
Means: [ 50.83001009  70.08920817  89.89619943 114.82257346 145.01706795
 162.94168929 199.01911223 237.4271152 ]
Processing 4/100
EM segmentation finished
Means: [ 53.17370389  73.50407242  87.19958602 102.88901912 116.67851228
 131.83683612 151.73703403 185.47885366]
Processing 5/100
EM segmentation finished
Means: [ 48.56994836  63.07875033  95.93137151 127.93816912 155.02225967
 182.01601372 213.64394526 246.0370639 ]
Processing 6/100
EM segmentation finished
Means: [ 33.7775775   56.12260868  76.4927524   98.4767457  124.40167836
 153.83276036 190.50897873 236.13681385]
Processing 7/100
EM segmentation finished
Means: [ 28.59756607  

In [42]:
def summarize(arr):
    return np.mean(arr), np.std(arr)

psnr_mean, psnr_std = summarize(psnr_list_)
ssim_mean, ssim_std = summarize(ssim_list_)
ent_mean, ent_std = summarize(entropy_change_list_)

print("\n===== POVM GMM =====")
print(f"PSNR: {psnr_mean:.4f} ± {psnr_std:.4f}")
print(f"SSIM: {ssim_mean:.4f} ± {ssim_std:.4f}")
print(f"Entropy Change (%): {ent_mean:.4f} ± {ent_std:.4f}")


===== POVM GMM =====
PSNR: 33.4993 ± 1.9285
SSIM: 0.9483 ± 0.0208
Entropy Change (%): -25.9031 ± 4.5221


In [ ]:
import os
import random
import cv2
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from skimage.measure import shannon_entropy

# metric lists
psnr_list = []
ssim_list = []
entropy_change_list = []

for idx, image_path in enumerate(sample_images):
    try:
        print(f"Processing {idx+1}/100")

        # Load image
        image = cv2.imread(image_path)

        # Step 1: grayscale
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

        # Step 2: CLAHE
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        clahe_image = clahe.apply(gray)

        # ---- Metrics ---- #

        psnr_value = peak_signal_noise_ratio(gray, clahe_image)

        ssim_value, _ = structural_similarity(gray, clahe_image, full=True)

        entropy_original = shannon_entropy(gray)
        entropy_clahe = shannon_entropy(clahe_image)

        percent_entropy_change = (
            (entropy_clahe - entropy_original) / entropy_original
        ) * 100

        # append to lists
        psnr_list.append(psnr_value)
        ssim_list.append(ssim_value)
        entropy_change_list.append(percent_entropy_change)

    except Exception as e:
        print(f"Error with {image_path}: {e}")

Processing 1/100
Processing 2/100
Processing 3/100
Processing 4/100
Processing 5/100
Processing 6/100
Processing 7/100
Processing 8/100
Processing 9/100
Processing 10/100
Processing 11/100
Processing 12/100
Processing 13/100
Processing 14/100
Processing 15/100
Processing 16/100
Processing 17/100
Processing 18/100
Processing 19/100
Processing 20/100
Processing 21/100
Processing 22/100
Processing 23/100
Processing 24/100
Processing 25/100
Processing 26/100
Processing 27/100
Processing 28/100
Processing 29/100
Processing 30/100
Processing 31/100
Processing 32/100
Processing 33/100
Processing 34/100
Processing 35/100
Processing 36/100
Processing 37/100
Processing 38/100
Processing 39/100
Processing 40/100
Processing 41/100
Processing 42/100
Processing 43/100
Processing 44/100
Processing 45/100
Processing 46/100
Processing 47/100
Processing 48/100
Processing 49/100
Processing 50/100
Processing 51/100
Processing 52/100
Processing 53/100
Processing 54/100
Processing 55/100
Processing 56/100
P

In [ ]:
def summarize(arr):
    return np.mean(arr), np.std(arr)

psnr_mean, psnr_std = summarize(psnr_list)
ssim_mean, ssim_std = summarize(ssim_list)
ent_mean, ent_std = summarize(entropy_change_list)

print("\n===== CLAHE =====")
print(f"PSNR: {psnr_mean:.4f} ± {psnr_std:.4f}")
print(f"SSIM: {ssim_mean:.4f} ± {ssim_std:.4f}")
print(f"Entropy Change (%): {ent_mean:.4f} ± {ent_std:.4f}")


===== CLAHE =====
PSNR: 20.2173 ± 2.1111
SSIM: 0.8373 ± 0.0438
Entropy Change (%): 6.4902 ± 4.5446


In [ ]:
%pip install bm3d --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 862.0/862.0 kB 13.9 MB/s eta 0:00:00


In [ ]:
#BM3D
import os
import random
import cv2
import numpy as np
from bm3d import bm3d
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from skimage.measure import shannon_entropy

psnr_list = []
ssim_list = []
entropy_change_list = []

for idx, image_path in enumerate(sample_images):
    try:
        print(f"Processing {idx+1}/100")

        image = cv2.imread(image_path)
        if image is None:
            continue

        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

        # normalize to [0,1] for bm3d
        gray_norm = gray / 255.0

        # BM3D
        denoised = bm3d(gray_norm, sigma_psd=0.1)

        # back to uint8
        denoised = (denoised * 255).astype(np.uint8)

        # metrics
        psnr = peak_signal_noise_ratio(gray, denoised)
        ssim, _ = structural_similarity(gray, denoised, full=True)

        ent_orig = shannon_entropy(gray)
        ent_new = shannon_entropy(denoised)

        entropy_change = ((ent_new - ent_orig) / ent_orig) * 100

        psnr_list.append(psnr)
        ssim_list.append(ssim)
        entropy_change_list.append(entropy_change)

    except Exception as e:
        print(f"Error: {e}")

# ---- stats ---- #
import numpy as np

print("\n===== BM3D =====")
print(f"PSNR: {np.mean(psnr_list):.4f} ± {np.std(psnr_list):.4f}")
print(f"SSIM: {np.mean(ssim_list):.4f} ± {np.std(ssim_list):.4f}")
print(f"Entropy Change (%): {np.mean(entropy_change_list):.4f} ± {np.std(entropy_change_list):.4f}")

Processing 1/100
Processing 2/100
Processing 3/100
Processing 4/100
Processing 5/100
Processing 6/100
Processing 7/100
Processing 8/100
Processing 9/100
Processing 10/100
Processing 11/100
Processing 12/100
Processing 13/100
Processing 14/100
Processing 15/100
Processing 16/100
Processing 17/100
Processing 18/100
Processing 19/100
Processing 20/100
Processing 21/100
Processing 22/100
Processing 23/100
Processing 24/100
Processing 25/100
Processing 26/100
Processing 27/100
Processing 28/100
Processing 29/100
Processing 30/100
Processing 31/100
Processing 32/100
Processing 33/100
Processing 34/100
Processing 35/100
Processing 36/100
Processing 37/100
Processing 38/100
Processing 39/100
Processing 40/100
Processing 41/100
Processing 42/100
Processing 43/100
Processing 44/100
Processing 45/100
Processing 46/100
Processing 47/100
Processing 48/100
Processing 49/100
Processing 50/100
Processing 51/100
Processing 52/100
Processing 53/100
Processing 54/100
Processing 55/100
Processing 56/100
P

In [ ]:
import os
import random
import cv2
import numpy as np
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from skimage.measure import shannon_entropy

psnr_list = []
ssim_list = []
entropy_change_list = []

for idx, image_path in enumerate(sample_images):
    try:
        print(f"Processing {idx+1}/100")

        image = cv2.imread(image_path)
        if image is None:
            continue

        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

        gaussian_img = cv2.GaussianBlur(gray, (5, 5), sigmaX=1.0)

        # ---- metrics ---- #
        psnr = peak_signal_noise_ratio(gray, gaussian_img)
        ssim, _ = structural_similarity(gray, gaussian_img, full=True)

        ent_orig = shannon_entropy(gray)
        ent_new = shannon_entropy(gaussian_img)

        entropy_change = ((ent_new - ent_orig) / ent_orig) * 100

        psnr_list.append(psnr)
        ssim_list.append(ssim)
        entropy_change_list.append(entropy_change)

    except Exception as e:
        print(f"Error: {e}")

# ---- stats ---- #
print("\n===== Gaussian Filter =====")
print(f"PSNR: {np.mean(psnr_list):.4f} ± {np.std(psnr_list):.4f}")
print(f"SSIM: {np.mean(ssim_list):.4f} ± {np.std(ssim_list):.4f}")
print(f"Entropy Change (%): {np.mean(entropy_change_list):.4f} ± {np.std(entropy_change_list):.4f}")

Processing 1/100
Processing 2/100
Processing 3/100
Processing 4/100
Processing 5/100
Processing 6/100
Processing 7/100
Processing 8/100
Processing 9/100
Processing 10/100
Processing 11/100
Processing 12/100
Processing 13/100
Processing 14/100
Processing 15/100
Processing 16/100
Processing 17/100
Processing 18/100
Processing 19/100
Processing 20/100
Processing 21/100
Processing 22/100
Processing 23/100
Processing 24/100
Processing 25/100
Processing 26/100
Processing 27/100
Processing 28/100
Processing 29/100
Processing 30/100
Processing 31/100
Processing 32/100
Processing 33/100
Processing 34/100
Processing 35/100
Processing 36/100
Processing 37/100
Processing 38/100
Processing 39/100
Processing 40/100
Processing 41/100
Processing 42/100
Processing 43/100
Processing 44/100
Processing 45/100
Processing 46/100
Processing 47/100
Processing 48/100
Processing 49/100
Processing 50/100
Processing 51/100
Processing 52/100
Processing 53/100
Processing 54/100
Processing 55/100
Processing 56/100
P

In [24]:
# Bilateral Filter
import os
import random
import cv2
import numpy as np
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from skimage.measure import shannon_entropy

psnr_list = []
ssim_list = []
entropy_change_list = []

for idx, image_path in enumerate(sample_images):
    try:
        print(f"Processing {idx+1}/{len(sample_images)}")

        image = cv2.imread(image_path)
        if image is None:
            continue

        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

        # ---- Bilateral Filter ----
        # d = diameter of pixel neighborhood
        # sigmaColor = filter sigma in color space
        # sigmaSpace = filter sigma in coordinate space
        denoised = cv2.bilateralFilter(
            gray,
            d=5,
            sigmaColor=50,
            sigmaSpace=50
        )

        # ---- Metrics ----
        psnr = peak_signal_noise_ratio(gray, denoised)
        ssim, _ = structural_similarity(gray, denoised, full=True)

        ent_orig = shannon_entropy(gray)
        ent_new = shannon_entropy(denoised)

        entropy_change = ((ent_new - ent_orig) / ent_orig) * 100

        psnr_list.append(psnr)
        ssim_list.append(ssim)
        entropy_change_list.append(entropy_change)

    except Exception as e:
        print(f"Error: {e}")

# ---- stats ---- #
print("\n===== Bilateral Filter =====")
print(f"PSNR: {np.mean(psnr_list):.4f} ± {np.std(psnr_list):.4f}")
print(f"SSIM: {np.mean(ssim_list):.4f} ± {np.std(ssim_list):.4f}")
print(f"Entropy Change (%): {np.mean(entropy_change_list):.4f} ± {np.std(entropy_change_list):.4f}")

Processing 1/100
Processing 2/100
Processing 3/100
Processing 4/100
Processing 5/100
Processing 6/100
Processing 7/100
Processing 8/100
Processing 9/100
Processing 10/100
Processing 11/100
Processing 12/100
Processing 13/100
Processing 14/100
Processing 15/100
Processing 16/100
Processing 17/100
Processing 18/100
Processing 19/100
Processing 20/100
Processing 21/100
Processing 22/100
Processing 23/100
Processing 24/100
Processing 25/100
Processing 26/100
Processing 27/100
Processing 28/100
Processing 29/100
Processing 30/100
Processing 31/100
Processing 32/100
Processing 33/100
Processing 34/100
Processing 35/100
Processing 36/100
Processing 37/100
Processing 38/100
Processing 39/100
Processing 40/100
Processing 41/100
Processing 42/100
Processing 43/100
Processing 44/100
Processing 45/100
Processing 46/100
Processing 47/100
Processing 48/100
Processing 49/100
Processing 50/100
Processing 51/100
Processing 52/100
Processing 53/100
Processing 54/100
Processing 55/100
Processing 56/100
P

In [25]:
# Fast Non-Local Means (OpenCV)
import os
import random
import cv2
import numpy as np
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from skimage.measure import shannon_entropy

psnr_list = []
ssim_list = []
entropy_change_list = []

for idx, image_path in enumerate(sample_images):
    try:
        print(f"Processing {idx+1}/{len(sample_images)}")

        image = cv2.imread(image_path)
        if image is None:
            continue

        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

        # ---- Fast NLM ----
        # h = filter strength (higher = stronger denoising)
        # templateWindowSize = patch size
        # searchWindowSize = area to search similar patches
        denoised = cv2.fastNlMeansDenoising(
            gray,
            None,
            h=10,
            templateWindowSize=7,
            searchWindowSize=21
        )

        # ---- Metrics ----
        psnr = peak_signal_noise_ratio(gray, denoised)
        ssim, _ = structural_similarity(gray, denoised, full=True)

        ent_orig = shannon_entropy(gray)
        ent_new = shannon_entropy(denoised)

        entropy_change = ((ent_new - ent_orig) / ent_orig) * 100

        psnr_list.append(psnr)
        ssim_list.append(ssim)
        entropy_change_list.append(entropy_change)

    except Exception as e:
        print(f"Error: {e}")

# ---- stats ---- #
print("\n===== FastNLM (OpenCV) =====")
print(f"PSNR: {np.mean(psnr_list):.4f} ± {np.std(psnr_list):.4f}")
print(f"SSIM: {np.mean(ssim_list):.4f} ± {np.std(ssim_list):.4f}")
print(f"Entropy Change (%): {np.mean(entropy_change_list):.4f} ± {np.std(entropy_change_list):.4f}")

Processing 1/100
Processing 2/100
Processing 3/100
Processing 4/100
Processing 5/100
Processing 6/100
Processing 7/100
Processing 8/100
Processing 9/100
Processing 10/100
Processing 11/100
Processing 12/100
Processing 13/100
Processing 14/100
Processing 15/100
Processing 16/100
Processing 17/100
Processing 18/100
Processing 19/100
Processing 20/100
Processing 21/100
Processing 22/100
Processing 23/100
Processing 24/100
Processing 25/100
Processing 26/100
Processing 27/100
Processing 28/100
Processing 29/100
Processing 30/100
Processing 31/100
Processing 32/100
Processing 33/100
Processing 34/100
Processing 35/100
Processing 36/100
Processing 37/100
Processing 38/100
Processing 39/100
Processing 40/100
Processing 41/100
Processing 42/100
Processing 43/100
Processing 44/100
Processing 45/100
Processing 46/100
Processing 47/100
Processing 48/100
Processing 49/100
Processing 50/100
Processing 51/100
Processing 52/100
Processing 53/100
Processing 54/100
Processing 55/100
Processing 56/100
P